In [ ]:
# ============================================================
# GOLD FACTOR RESEARCH
# NOTEBOOK 04 - MODELING AND BACKTESTING
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

plt.style.use("ggplot")

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    "../data/processed/factor_data.csv",
    index_col=0,
    parse_dates=True
)

print("Dataset Shape:", df.shape)

# ============================================================
# MODEL FEATURES
# ============================================================

features = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "above_200ma",
    "vol20",
    "gold_vs_spy",
    "gold_vs_silver"
]

target = "target"

# ============================================================
# DROP MISSING VALUES
# ============================================================

model_df = df[
    features + [target]
].dropna()

print("Model Dataset Shape:")
print(model_df.shape)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

split_index = int(
    len(model_df) * 0.8
)

train_df = model_df.iloc[:split_index]

test_df = model_df.iloc[split_index:]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

# ============================================================
# LOGISTIC REGRESSION
# ============================================================

log_model = LogisticRegression(
    max_iter=1000
)

log_model.fit(
    X_train,
    y_train
)

log_preds = log_model.predict(
    X_test
)

log_probs = (
    log_model
    .predict_proba(X_test)
)[:,1]

# ============================================================
# RANDOM FOREST
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

rf_preds = rf_model.predict(
    X_test
)

rf_probs = (
    rf_model
    .predict_proba(X_test)
)[:,1]

# ============================================================
# XGBOOST
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_preds = xgb_model.predict(
    X_test
)

xgb_probs = (
    xgb_model
    .predict_proba(X_test)
)[:,1]

# ============================================================
# MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    y_true,
    predictions,
    probabilities
):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions
        ),
        "Recall": recall_score(
            y_true,
            predictions
        ),
        "F1": f1_score(
            y_true,
            predictions
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities
        )
    }

# ============================================================
# EVALUATE MODELS
# ============================================================

results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        log_preds,
        log_probs
    ),

    evaluate_model(
        "Random Forest",
        y_test,
        rf_preds,
        rf_probs
    ),

    evaluate_model(
        "XGBoost",
        y_test,
        xgb_preds,
        xgb_probs
    )
])

results = results.sort_values(
    by="ROC_AUC",
    ascending=False
)

print("\nMODEL PERFORMANCE")
display(results)

# ============================================================
# SAVE MODEL RESULTS
# ============================================================

results.to_csv(
    "../reports/model_results.csv",
    index=False
)

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    "Feature": features,

    "Importance":
    xgb_model.feature_importances_

})

importance_df = (
    importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
)

display(
    importance_df
)

# ============================================================
# FEATURE IMPORTANCE CHART
# ============================================================

plt.figure(
    figsize=(10,6)
)

plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)

plt.title(
    "XGBoost Feature Importance"
)

plt.tight_layout()

plt.savefig(
    "../images/feature_importance.png"
)

plt.show()

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    xgb_preds
)

plt.figure(
    figsize=(6,5)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    "XGBoost Confusion Matrix"
)

plt.savefig(
    "../images/confusion_matrix.png"
)

plt.show()

# ============================================================
# BACKTEST PREPARATION
# ============================================================

full_df = model_df.copy()

full_df["probability"] = (
    xgb_model
    .predict_proba(
        full_df[features]
    )
)[:,1]

# ============================================================
# GENERATE SIGNALS
# ============================================================

full_df["signal"] = np.where(
    full_df["probability"] > 0.60,
    1,
    0
)

# ============================================================
# GOLD RETURNS
# ============================================================

df["gold_return"] = (
    df["gold"]
    .pct_change()
)

full_df["gold_return"] = (
    df.loc[
        full_df.index,
        "gold_return"
    ]
)

# ============================================================
# STRATEGY RETURNS
# ============================================================

full_df["strategy_return"] = (

    full_df["signal"]
    .shift(1)
    .fillna(0)

    *

    full_df["gold_return"]
)

# ============================================================
# BUY AND HOLD
# ============================================================

full_df["buy_hold"] = (
    full_df["gold_return"]
)

# ============================================================
# EQUITY CURVES
# ============================================================

full_df["strategy_equity"] = (

    1 +
    full_df["strategy_return"]
).cumprod()

full_df["buyhold_equity"] = (

    1 +
    full_df["buy_hold"]
).cumprod()

# ============================================================
# EQUITY CURVE CHART
# ============================================================

plt.figure(
    figsize=(12,6)
)

plt.plot(
    full_df["strategy_equity"],
    label="Strategy"
)

plt.plot(
    full_df["buyhold_equity"],
    label="Buy & Hold Gold"
)

plt.legend()

plt.title(
    "Strategy vs Buy & Hold"
)

plt.tight_layout()

plt.savefig(
    "../images/equity_curve.png"
)

plt.show()

# ============================================================
# PERFORMANCE METRICS
# ============================================================

def annual_return(returns):

    cumulative = (
        1 + returns
    ).prod()

    years = (
        len(returns)
        / 252
    )

    return (
        cumulative **
        (1 / years)
        - 1
    )

def sharpe_ratio(returns):

    return (
        returns.mean()
        /
        returns.std()
    ) * np.sqrt(252)

def sortino_ratio(returns):

    downside = (
        returns[
            returns < 0
        ]
        .std()
    )

    return (
        returns.mean()
        /
        downside
    ) * np.sqrt(252)

def max_drawdown(returns):

    equity = (
        1 + returns
    ).cumprod()

    peak = (
        equity.cummax()
    )

    drawdown = (
        equity
        /
        peak
        - 1
    )

    return drawdown.min()

# ============================================================
# STRATEGY STATISTICS
# ============================================================

strategy_returns = (
    full_df["strategy_return"]
    .dropna()
)

stats = {

    "Annual Return":
    annual_return(
        strategy_returns
    ),

    "Sharpe Ratio":
    sharpe_ratio(
        strategy_returns
    ),

    "Sortino Ratio":
    sortino_ratio(
        strategy_returns
    ),

    "Maximum Drawdown":
    max_drawdown(
        strategy_returns
    ),

    "Win Rate":
    (
        strategy_returns > 0
    ).mean()
}

stats_df = pd.DataFrame(
    stats,
    index=["Strategy"]
)

display(stats_df)

# ============================================================
# MONTE CARLO SIMULATION
# ============================================================

num_simulations = 100

num_days = 252

mu = strategy_returns.mean()

sigma = strategy_returns.std()

plt.figure(
    figsize=(12,6)
)

for _ in range(
    num_simulations
):

    simulation = np.random.normal(
        mu,
        sigma,
        num_days
    )

    path = (
        1 + simulation
    ).cumprod()

    plt.plot(
        path,
        alpha=0.20
    )

plt.title(
    "Monte Carlo Simulation"
)

plt.tight_layout()

plt.savefig(
    "../images/monte_carlo.png"
)

plt.show()

# ============================================================
# CURRENT FORECAST
# ============================================================

latest_data = (
    full_df[features]
    .iloc[[-1]]
)

current_probability = (

    xgb_model
    .predict_proba(
        latest_data
    )[0][1]

)

print(
    "\nCurrent Probability Gold Gains >5% In 5 Months:"
)

print(
    f"{current_probability:.2%}"
)

# ============================================================
# EXPORT RESULTS
# ============================================================

stats_df.to_csv(
    "../reports/backtest_results.csv"
)

full_df.to_csv(
    "../data/processed/model_predictions.csv"
)

print(
    "\nFiles Saved:"
)

print(
    "- model_results.csv"
)

print(
    "- backtest_results.csv"
)

print(
    "- model_predictions.csv"
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*60)

print(
    "MODELING AND BACKTEST COMPLETE"
)

print("="*60)

print("""

Outputs Generated:

✓ Logistic Regression

✓ Random Forest

✓ XGBoost

✓ Model Comparison

✓ Feature Importance

✓ Confusion Matrix

✓ Strategy Backtest

✓ Equity Curve

✓ Monte Carlo Simulation

✓ Current Forecast Probability

✓ Exported Results

Images Saved:

- feature_importance.png

- confusion_matrix.png

- equity_curve.png

- monte_carlo.png

Next Step:

dashboard/app.py

Use exported results to build
the interactive Streamlit dashboard.

""")